# 00b — Baseline univariado: Oxigênio Dissolvido da estação EF01 (CETESB)

**Objetivo:** repetir o protocolo do exp 00 no Oxigênio Dissolvido (OD, mg/L).
**Decisões travadas:** variável `OD` · lookback `L=8640` (30 dias) · horizonte `H=288` (1 dia) · split temporal 70/15/15 pré-holdout sem shuffle · **holdout puro nos últimos 10 dias do segmento limpo** · baselines clássicos (persistência, sazonal-naive, média móvel, ARIMA em grade horária, Prophet).
**Recorte necessário:** o sensor de OD ficou morto de 21/07 01:10 a 06/08 11:30 (16,4 dias, ver §2) — gap impossível de interpolar. O experimento usa o **segmento limpo 01/06 → 21/07** (50,0 dias); o pós-gap (24,5 dias) é curto demais para `L=30d` e fica como trabalho futuro.
**Dados:** `dados/ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv` — ver `dados/README.md`.

In [1]:
import pickle
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

# --- caminhos (funciona com cwd = repo ou notebooks/) ---
ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
CSV = ROOT / "dados" / "ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv"
OUT = ROOT / "resultados" / "00b-baseline-od"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# --- hiperparâmetros do experimento (iguais ao 00) ---
L, H = 8640, 288           # lookback (30 dias) e horizonte (1 dia) em passos de 5 min
SEASON = 288              # ciclo diário em passos de 5 min
INTERP_LIMIT = 24         # interpolação temporal máx. (24 passos = 2 h)
SEG_FIM = "2026-07-21 01:05"  # fim do segmento limpo (início do gap de 16,4 dias)
HOLDOUT_DIAS = 10          # cauda final do SEGMENTO separada como holdout puro (só alvos)
ARIMA_ORDER = (2, 1, 2)   # ARIMA roda em grade horária (L=720h, H=24h)
ARIMA_STRIDE = 24         # ARIMA reestimado a cada 24 origens do teste rolante

print("ROOT:", ROOT, "| CSV existe:", CSV.exists())

ROOT: /home/marcos/Projetos/temporal-model | CSV existe: True


## 1. Carga
Formato CETESB: `;`, decimal com vírgula, `windows-1252`, linha 1 = validação, linha 2 = cabeçalho.

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
colvar = [c for c in df.columns if c != "Data hora"][0]
df = df.rename(columns={"Data hora": "ds", colvar: "y"}).sort_values("ds").reset_index(drop=True)
print(colvar, "|", df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

Oxigênio Dissolvido (mg/L) | (26209, 2) 2026-06-01 00:00:00 → 2026-08-31 00:00:00
faltantes: 4767 (18.2%)


,ds,y
count,26209,21442.000000
mean,2026-07-16 12:00:00,6.668501
min,2026-06-01 00:00:00,4.840000
25%,2026-06-23 18:00:00,6.320000
50%,2026-07-16 12:00:00,6.630000
75%,2026-08-08 06:00:00,6.950000
max,2026-08-31 00:00:00,8.910000
std,NaN,0.575499


## 2. EDA — perfil, o gap de 16 dias e ciclo diário

In [3]:
isna = df["y"].isna().to_numpy()
bounds = np.where(np.diff(np.concatenate([[False], isna, [False]])))[0]
runs = sorted([(bounds[i], bounds[i+1]-1) for i in range(0, len(bounds), 2)],
              key=lambda r: r[1]-r[0], reverse=True)
print("top 5 gaps:")
for a, b in runs[:5]:
    print(f"  {df.ds[a]} → {df.ds[b]}  ({(b-a+1)*5/60:.1f} h)")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.4)
ax[0].axvspan(pd.Timestamp("2026-07-21 01:10"), pd.Timestamp("2026-08-06 11:30"),
              color="r", alpha=0.2, label="sensor morto (16,4 dias)")
ax[0].set_title("OD EF01 — série completa (faixa vermelha = gap, fora do experimento)")
ax[0].set_ylabel("OD (mg/L)")
ax[0].legend(fontsize=8)
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição do OD")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("OD por hora do dia (ciclo diário?)")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva:", OUT / "figs" / "01-eda.png")

top 5 gaps:
  2026-07-21 01:10:00 → 2026-08-06 11:30:00  (394.4 h)
  2026-06-30 22:05:00 → 2026-06-30 23:55:00  (1.9 h)
  2026-07-11 10:15:00 → 2026-07-11 10:30:00  (0.3 h)
  2026-07-16 09:40:00 → 2026-07-16 09:45:00  (0.2 h)
  2026-07-20 20:10:00 → 2026-07-20 20:10:00  (0.1 h)


fig salva: /home/marcos/Projetos/temporal-model/resultados/00b-baseline-od/figs/01-eda.png


## 3. Limpeza + recorte do segmento limpo
Grade de 5 min, interpolação máx. 2 h e **corte em 21/07 01:05** (antes do gap). Tudo a jusante usa só o segmento 01/06 → 21/07.

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_full = df.set_index("ds")["y"].reindex(idx)
s = s_full.loc[:SEG_FIM].interpolate(method="time", limit=INTERP_LIMIT)
print(f"segmento: {s.index.min()} → {s.index.max()} ({len(s)} slots = {len(s)*5/60/24:.1f} dias)")
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")

amostra = slice("2026-06-08", "2026-06-15")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_full[amostra].index, s_full[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 08–15/06")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

segmento: 2026-06-01 00:00:00 → 2026-07-21 01:05:00 (14414 slots = 50.0 dias)
NaN após interpolação (limite 24): 0


fig salva


## 4. Estacionariedade (ADF) e decomposição STL

In [5]:
n_total = len(s)
n_train = int(n_total * 0.70)
train = s.iloc[:n_train].dropna()
stat, pval, *_ = adfuller(train.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária (usar diferenciação / modelos robustos)'}")

stl = STL(train.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-4.93 p-valor=3.03e-05 → estacionária


fig salva


## 5. Janelamento + holdout puro (dentro do segmento)
Amostras `(L=8640 → H=288)`, só janelas 100% observadas. Pré-holdout: split 70/15/15 **sem shuffle**. Holdout: últimos 10 dias do segmento — **alvos nunca treinados** — mais 10 origens diárias.

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
n = len(X)
ZONE = s.index.max() - pd.Timedelta(days=HOLDOUT_DIAS)
is_hold = ends >= (ZONE + pd.Timedelta(minutes=5 * (H - 1)))
ho = np.where(is_hold)[0]
pre = np.where(~is_hold)[0]
i1, i2 = int(len(pre) * 0.70), int(len(pre) * 0.85)
tr, va, te = pre[:i1], pre[i1:i2], pre[i2:]
splits = {"train": tr, "val": va, "test": te, "holdout": ho}
for k, idx in splits.items():
    print(f"{k}: {len(idx)} janelas | alvos {ends[idx[0]].date()} → {ends[idx[-1]].date()}")
print(f"janelas descartadas (com NaN): {len(s) - L - H + 1 - n}")
print(f"zona holdout (alvos): {ZONE.date()} → {s.index.max().date()}")
daily_ends = [ZONE + pd.Timedelta(minutes=5 * (H - 1 + H * k)) for k in range(HOLDOUT_DIAS)]
daily_idx = np.array([int(np.where(ends == d)[0][0]) for d in daily_ends])
print("dias previstos:", [str(ends[i].date()) for i in daily_idx])
TR_END = ends[tr[-1]]

train: 2025 janelas | alvos 2026-07-01 → 2026-07-09
val: 434 janelas | alvos 2026-07-09 → 2026-07-10
test: 434 janelas | alvos 2026-07-10 → 2026-07-12
holdout: 2594 janelas | alvos 2026-07-12 → 2026-07-21
janelas descartadas (com NaN): 0
zona holdout (alvos): 2026-07-11 → 2026-07-21
dias previstos: ['2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16', '2026-07-17', '2026-07-18', '2026-07-19', '2026-07-20', '2026-07-21']


## 6. Baselines baratos (teste rolante + holdout)

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xte, Yte = X[te], Y[te]
Xho, Yho = X[ho], Y[ho]
pred_te = cheap_preds(Xte)
pred_ho = cheap_preds(Xho)
print("teste rolante:")
print(pd.DataFrame({m: metricas(Yte, p) for m, p in pred_te.items()}).T.round(4).to_string())
print("holdout (todas as origens):")
print(pd.DataFrame({m: metricas(Yho, p) for m, p in pred_ho.items()}).T.round(4).to_string())

teste rolante:


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.2371  0.3019  3.3575  3.3530
sazonal_naive_288  0.1525  0.1770  2.1668  2.1918
media_movel_288    0.1900  0.2468  2.6526  2.6942
holdout (todas as origens):
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.4633  0.6224  6.1170  6.1055
sazonal_naive_288  0.1509  0.2194  2.0324  2.0591
media_movel_288    0.3952  0.4757  5.1949  5.2617


## 7. ARIMA em grade horária (subamostras — custo)
ARIMA(2,1,2) sobre a série reamostrada para 1 h (`L=720h`, `H=24h`); cada previsão horária é repetida 12× (aproximação documentada).

In [8]:
hs = s.resample("1h").mean()

def arima_hora(e):
    he = e.floor("h")
    ctx = hs.loc[he - pd.Timedelta(hours=719):he].values
    fc = ARIMA(ctx, order=ARIMA_ORDER).fit().get_forecast(24).predicted_mean.values
    return np.repeat(fc, 12)[:H]

def roda_arima(idxs, nome):
    P = np.empty((len(idxs), H))
    t0 = time.time()
    for j, i in enumerate(idxs):
        try:
            P[j] = arima_hora(ends[i])
        except Exception:
            P[j] = np.repeat(X[i, -1], H)
        if (j + 1) % 10 == 0:
            print(f"  {nome}: {j+1}/{len(idxs)} origens...", flush=True)
    print(f"ARIMA {nome}: {len(idxs)} origens em {time.time()-t0:.0f}s")
    return P

idx_a = np.arange(0, len(te), ARIMA_STRIDE)
Pa = roda_arima(te[idx_a], "teste")
print(metricas(Y[te[idx_a]], Pa))
Pd = roda_arima(daily_idx, "holdout-diario")
print("holdout diário:", metricas(Y[daily_idx], Pd))

h_tr = hs.loc[:TR_END].iloc[-720:].values
with open(OUT / "modelos" / "arima212_cauda_treino.pkl", "wb") as f:
    pickle.dump(ARIMA(h_tr, order=ARIMA_ORDER).fit(), f)
print("modelo salvo")

  teste: 10/19 origens...


ARIMA teste: 19 origens em 25s
{'MAE': 0.2381988304093567, 'RMSE': 0.30163624204432044, 'MAPE': 3.37381573440214, 'sMAPE': 3.370081523641914}


  holdout-diario: 10/10 origens...


ARIMA holdout-diario: 10 origens em 12s
holdout diário: {'MAE': 0.4272881944444445, 'RMSE': 0.49209413228656185, 'MAPE': 5.703635972787773, 'sMAPE': 5.668510264711123}


modelo salvo


## 8. Prophet (opcional — pula se `prophet`/CmdStan indisponível)

In [9]:
PROPHET_OK = False
try:
    from prophet import Prophet
    import cmdstanpy
    assert cmdstanpy.cmdstan_path() is not None
    df_train = pd.DataFrame({"ds": s.loc[:TR_END].index, "y": s.loc[:TR_END].values}).dropna()
    m = Prophet(daily_seasonality=True, weekly_seasonality=True)
    m.fit(df_train)
    fmap = m.predict(pd.DataFrame({"ds": s.index})).set_index("ds")["yhat"]

    def fatia(idxs):
        E = ends[idxs]
        return np.stack([[fmap.loc[d - pd.Timedelta(minutes=5*(H-1-h))] for h in range(H)] for d in E])

    Pp_te, Pp_ho, Pp_d = fatia(te), fatia(ho), fatia(daily_idx)
    PROPHET_OK = True
    print("Prophet teste:", metricas(Yte, Pp_te))
    print("Prophet holdout:", metricas(Yho, Pp_ho))
    from prophet.serialize import model_to_json
    (OUT / "modelos" / "prophet_od.json").write_text(model_to_json(m))
    print("modelo salvo")
except Exception as e:
    print(f"Prophet pulado ({type(e).__name__}: {str(e)[:150]}).")

Importing plotly failed. Interactive plots will not work.


19:48:20 - cmdstanpy - INFO - Chain [1] start processing


19:48:34 - cmdstanpy - INFO - Chain [1] done processing


Prophet teste: {'MAE': 0.20511679642798802, 'RMSE': 0.2285045697530959, 'MAPE': 2.9603152653895197, 'sMAPE': 2.909357824407486}
Prophet holdout: {'MAE': 0.36798296346050724, 'RMSE': 0.4406951143349837, 'MAPE': 4.929755721567689, 'sMAPE': 4.897179770240742}
modelo salvo


## 9. Comparação final + holdout dia a dia

In [10]:
linhas = {m: metricas(Yte, p) for m, p in pred_te.items()}
if PROPHET_OK:
    linhas["prophet"] = metricas(Yte, Pp_te)
tab = pd.DataFrame(linhas).T.round(4)
tab.to_csv(OUT / "metricas_baseline.csv")
print("=== teste rolante ===")
print(tab.to_string())

Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in pred_te}
diario["arima_212_h"] = metricas(Yd, Pd)
if PROPHET_OK:
    diario["prophet"] = metricas(Yd, Pp_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_holdout.csv")
print("=== holdout diário (10 dias) ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in pred_te},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["arima_212_h"] = [mae(Yd[k:k+1], Pd[k:k+1]) for k in range(len(Yd))]
if PROPHET_OK:
    por_dia["prophet"] = [mae(Yd[k:k+1], Pp_d[k:k+1]) for k in range(len(Yd))]
print(por_dia.round(4).to_string())
print(f"\nRégua (menor MAE no teste rolante): {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}")
print(f"Régua (menor MAE no holdout diário): {tab_d['MAE'].idxmin()} = {tab_d['MAE'].min():.4f}")

=== teste rolante ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.2371  0.3019  3.3575  3.3530
sazonal_naive_288  0.1525  0.1770  2.1668  2.1918
media_movel_288    0.1900  0.2468  2.6526  2.6942
prophet            0.2051  0.2285  2.9603  2.9094
=== holdout diário (10 dias) ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.4273  0.4921  5.7036  5.6685
sazonal_naive_288  0.1550  0.2233  2.0794  2.1009
media_movel_288    0.4071  0.4916  5.3416  5.4047
arima_212_h        0.4273  0.4921  5.7036  5.6685
prophet            0.3745  0.4469  5.0083  4.9765


            persistencia  sazonal_naive_288  media_movel_288  arima_212_h  prophet
2026-07-12        0.2965             0.0762           0.2350       0.2965   0.2147
2026-07-13        0.4225             0.1595           0.2799       0.4225   0.3799
2026-07-14        0.3428             0.2452           0.3492       0.3428   0.4499
2026-07-15        0.3178             0.4544           0.4544       0.3178   0.2347
2026-07-16        0.3478             0.1219           0.3066       0.3478   0.2182
2026-07-17        0.3919             0.0647           0.3812       0.3919   0.3084
2026-07-18        0.4505             0.0592           0.4293       0.4505   0.4112
2026-07-19        0.4940             0.0620           0.4790       0.4940   0.4362
2026-07-20        0.5772             0.0880           0.5472       0.5772   0.4856
2026-07-21        0.6319             0.2188           0.6095       0.6319   0.6059

Régua (menor MAE no teste rolante): sazonal_naive_288 = 0.1525
Régua (menor MAE no hol

In [11]:
E = ends[te]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, [0, len(Xte)//2, -1]):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xte[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Yte[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_te["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, pred_te["persistencia"][k], ":", lw=1, label="persistência")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE no teste rolante — baselines (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharey=False)
for ax, k in zip(axes.ravel(), range(len(Yd))):
    tf = pd.date_range(ends[daily_idx[k]] - pd.Timedelta(minutes=5*(H-1)), ends[daily_idx[k]], freq="5min")
    ax.plot(tf, Yd[k], "k-", lw=1.2, label="real")
    ax.plot(tf, cheap_preds(X[daily_idx])["persistencia"][k], ":", lw=1, label="persistência")
    ax.plot(tf, cheap_preds(X[daily_idx])["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    if PROPHET_OK:
        ax.plot(tf, Pp_d[k], "-.", lw=1, label="prophet")
    ax.plot(tf, Pd[k], lw=1, alpha=0.7, label="arima-h")
    ax.set_title(f"dia previsto {ends[daily_idx[k]].date()} (MAE pers={por_dia['persistencia'].iloc[k]:.3f})")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-holdout-dias.png")
print("figs salvas")

figs salvas


## 10. Conclusões e próximos passos

- A **régua** (menor MAE, impressa na §9) vale para teste rolante e holdout separadamente: o OD tem amplitude e ciclo maiores que o pH — ver se a hierarquia dos modelos muda.
- ARIMA roda em grade horária com expansão ×12 (aproximação documentada na §7); compare-o só na tabela diária.
- O pós-gap (06/08 → 31/08) ficou fora por falta de contexto de 30 dias — candidato a experimento de transferência/robustez.
- Artefatos em `resultados/00b-baseline-od/`: `metricas_baseline.csv`, `metricas_holdout.csv`, `modelos/` (ARIMA + Prophet) e `figs/` (inclui `06-holdout-dias.png`).